### Requirements

In [20]:
import pandas as pd
from scipy.stats import chisquare
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import mannwhitneyu

### Data

In [3]:
user_journey = pd.read_csv('../data/transformed_data/user_journey.csv')
experiment_results = pd.read_csv('../data/transformed_data/experiment_results.csv')
subscriptions = pd.read_csv('../data/transformed_data/subscriptions.csv')

### Test 1: SRM Check

In [10]:
# SRM (Simple Ratio Mismatch)
# This is a chi-square goodness-of-fit test to see how far the observed group counts are compared to the expected counts would be if the split was 50/50
# The two groups here are 'control' and 'treatment'

# observed counts
observed = experiment_results['variant'].value_counts()
control_n = observed['control']
treatment_n = observed['treatment']
total_n = control_n + treatment_n

# Expected counts under a true 50/50 split
expected = [total_n / 2, total_n / 2]
observed_counts = [control_n, treatment_n]

chi2_stat, p_value = chisquare(f_obs=observed_counts, f_exp=expected)

print(f"Control: {(control_n/total_n)*100:.2f}, Treatment: {(treatment_n/total_n)*100:.2f}")
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("SRM detected as group sizes significantly differ from 50/50")
else:
    print("No significant SRM detected")

Control: 51.97, Treatment: 48.03
Chi-square statistic: 15.5236
P-value: 0.0001
SRM detected as group sizes significantly differ from 50/50


In [ ]:
# SRM is statistically significant, but as the magnitude is 4 percentage points, it isn't as extreme (52% control and 48% treatment instead of 50% each)

### Test 2: Funnel Analysis

In [28]:
funnel_steps = ['did_verify_email', 'did_complete_profile', 'did_submit_kyc', 'did_get_kyc_approved', 'did_add_payment', 'did_first_transaction','did_engage_feature']

In [42]:
# Table 1: Cumulative conversion rate
# It shows what % of all users (not just those who reached the previous step) completed this step
cumulative = user_journey.groupby('variant')[funnel_steps].mean()

# transposing the data so the steps are rows and variants are columns
cumulative = cumulative.T

cumulative.columns = ['control_pct', 'treatment_pct']
cumulative['lift_pp'] = cumulative['treatment_pct'] - cumulative['control_pct']
cumulative *= 100
print("Cumulative funnel:")
print(cumulative.round(2))

Cumulative funnel:
                       control_pct  treatment_pct  lift_pp
did_verify_email             85.20          91.32     6.11
did_complete_profile         64.61          76.26    11.65
did_submit_kyc               41.29          54.86    13.57
did_get_kyc_approved         35.44          48.20    12.76
did_add_payment              25.25          37.60    12.36
did_first_transaction        16.86          25.98     9.13
did_engage_feature            9.31          15.59     6.28


In [44]:
# Table 2: Step-to-step conversion rate
# It shows what % of users who reached the previous step completed this step

step_to_step = pd.DataFrame(index=funnel_steps, columns=['control_pct', 'treatment_pct'])

prev_step = None
for step in funnel_steps:
    if prev_step is None:
        step_to_step.loc[step] = cumulative.loc[step, ['control_pct', 'treatment_pct']].values
    else:
        for variant, group in [('control_pct', 'control'), ('treatment_pct', 'treatment')]:
            prev_completed = user_journey[user_journey['variant'] == group][prev_step].sum()
            curr_completed = user_journey[user_journey['variant'] == group][step].sum()
            step_to_step.loc[step, variant] = (curr_completed / prev_completed * 100) if prev_completed > 0 else 0
    prev_step = step

step_to_step['lift_pp'] = step_to_step['treatment_pct'] - step_to_step['control_pct']
print("Step-to-step funnel:")
print(step_to_step.round(2))

Step-to-step funnel:
                      control_pct treatment_pct   lift_pp
did_verify_email        85.203002     91.317926  6.114925
did_complete_profile    75.835592     83.515732   7.68014
did_submit_kyc          63.907088     71.935572  8.028484
did_get_kyc_approved     85.83411     87.855787  2.021678
did_add_payment         71.226927     78.012959  6.786032
did_first_transaction   66.768293      69.10299  2.334697
did_engage_feature      55.251142     60.016026  4.764884


### Test 3: A/B test on primary metric (conversion rate)

In [ ]:
# The primary metric for the A/B test is conversion rate, i.e., whether someone did_first_transaction or not
# two-proportion z-test tests if the difference between two proportions (control conversion rate vs treatment conversion rate) is larger than what random chance would produce
# this test accounts for unequal group sizes


In [ ]:
control = experiment_results[experiment_results['variant'] == 'control']
treatment = experiment_results[experiment_results['variant'] == 'treatment']

control_conversions = control['did_first_transaction'].sum()
control_n = len(control)
treatment_conversions = treatment['did_first_transaction'].sum()
treatment_n = len(treatment)

counts = [control_conversions, treatment_conversions]
nobs = [control_n, treatment_n] # number of observations, i.e., the total number of users in each group

z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

control_rate = control_conversions / control_n
treatment_rate = treatment_conversions / treatment_n

print(f"Control conversion rate: {control_rate*100:.2f}")
print(f"Treatment conversion rate: {treatment_rate*100:.2f}")
print(f"Absolute lift: {(treatment_rate - control_rate)*100:.2f}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

In [58]:
control = experiment_results[experiment_results['variant'] == 'control']
treatment = experiment_results[experiment_results['variant'] == 'treatment']

control_conversions = control['did_first_transaction'].sum()
control_n = len(control)
treatment_conversions = treatment['did_first_transaction'].sum()
treatment_n = len(treatment)

counts = [control_conversions, treatment_conversions]
nobs = [control_n, treatment_n] # number of observations, i.e., the total number of users in each group

z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

control_rate = control_conversions / control_n
treatment_rate = treatment_conversions / treatment_n

print(f"Control conversion rate: {control_rate*100:.2f}")
print(f"Treatment conversion rate: {treatment_rate*100:.2f}")
print(f"Absolute lift: {(treatment_rate - control_rate)*100:.2f}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

Control conversion rate: 16.86
Treatment conversion rate: 25.98
Absolute lift: 9.13
Z-statistic: -11.1499
P-value: 0.000000
Statistically significant difference


### Test 4: A/B test on secondary metric (feature engagement)

In [ ]:
# The secondary metric for the A/B test is feature engagement, i.e., whether someone did_engage_feature or not
# This means they used at least one core feature of the app (checking their balance, setting up a payment, or using a budgeting tool)
# Two-proportions z-test

In [60]:
control = experiment_results[experiment_results['variant'] == 'control']
treatment = experiment_results[experiment_results['variant'] == 'treatment']

control_engaged = control['did_engage_feature'].sum()
control_n = len(control)
treatment_engaged = treatment['did_engage_feature'].sum()
treatment_n = len(treatment)

control_rate = control_engaged / control_n
treatment_rate = treatment_engaged / treatment_n

counts = [control_engaged, treatment_engaged]
nobs = [control_n, treatment_n]

z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

print(f"Control conversion rate: {control_rate*100:.2f}")
print(f"Treatment conversion rate: {treatment_rate*100:.2f}")
print(f"Absolute lift: {(treatment_rate - control_rate)*100:.2f}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

Control conversion rate: 9.31
Treatment conversion rate: 15.59
Absolute lift: 6.28
Z-statistic: -9.5451
P-value: 0.000000
Statistically significant difference


### Test 5: Notification CTR

In [ ]:
# Also a two-proportions z-test

In [7]:
control = experiment_results[experiment_results['variant'] == 'control']
treatment = experiment_results[experiment_results['variant'] == 'treatment']

control_clicks = control['notifications_clicked'].sum()
control_sent = control['notifications_sent'].sum()
treatment_clicks = treatment['notifications_clicked'].sum()
treatment_sent = treatment['notifications_sent'].sum()

control_ctr = control_clicks / control_sent
treatment_ctr = treatment_clicks / treatment_sent

counts = [control_clicks, treatment_clicks]
nobs = [control_sent, treatment_sent] # here the number of observations is the total notifications sent, not the total number of users as one user can receive multiple notifications

z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

print(f"Control CTR: {control_ctr*100:.2f}%")
print(f"Treatment CTR: {treatment_ctr*100:.2f}%")
print(f"Absolute lift: {(treatment_ctr - control_ctr)*100:.2f}pp")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

if p_value < 0.05:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

Control CTR: 21.81%
Treatment CTR: 28.42%
Absolute lift: 6.61pp
Z-statistic: -10.1053
P-value: 0.000000
Statistically significant difference


### Test 6: Segment Analysis

In [ ]:
# I am doing the segment analysis to check whether the effects of the treatment are consistent across different user groups (country, acquisition channel, and device type), instead of just looking at the overall average.
# The three segments are checked against the primary metric (conversion rate)

In [4]:
def run_ztest(df, metric_col):
    control = df[df['variant'] == 'control']
    treatment = df[df['variant'] == 'treatment']

    control_conv = control[metric_col].sum()
    control_n = len(control)
    treatment_conv = treatment[metric_col].sum()
    treatment_n = len(treatment)

    if control_n == 0 or treatment_n == 0:
        return None

    z_stat, p_value = proportions_ztest([control_conv, treatment_conv], [control_n, treatment_n])

    return {
        'control_n': control_n,
        'treatment_n': treatment_n,
        'control_rate': control_conv / control_n * 100,
        'treatment_rate': treatment_conv / treatment_n * 100,
        'lift_pp': (treatment_conv/treatment_n - control_conv/control_n) * 100,
        'p_value': p_value
    }

#### 6a. Segment by country

In [6]:
results = []
for country in experiment_results['country'].dropna().unique():
    segment = experiment_results[experiment_results['country'] == country]
    result = run_ztest(segment, 'did_first_transaction')
    if result:
        result['country'] = country
        results.append(result)

country_segments = pd.DataFrame(results).set_index('country')
country_segments['significant'] = country_segments['p_value'] < 0.05
print(country_segments.round(2))
print(f"\nSignificant in all countries: {country_segments['significant'].all()}")
print(f"Positive lift in all countries: {(country_segments['lift_pp'] > 0).all()}")

             control_n  treatment_n  control_rate  treatment_rate  lift_pp  \
country                                                                      
Germany           2663         2389         16.71           26.45     9.74   
Netherlands        772          712         16.45           23.46     7.00   
Switzerland        498          487         16.67           24.44     7.77   
Austria            774          726         17.44           27.96    10.52   
France             490          489         17.55           25.97     8.42   

             p_value  significant  
country                            
Germany          0.0         True  
Netherlands      0.0         True  
Switzerland      0.0         True  
Austria          0.0         True  
France           0.0         True  

Significant in all countries: True
Positive lift in all countries: True


#### 6b. Segment by acquisition channel

In [7]:
results = []
for acquisition_channel in experiment_results['acquisition_channel'].dropna().unique():
    segment = experiment_results[experiment_results['acquisition_channel'] == acquisition_channel]
    result = run_ztest(segment, 'did_first_transaction')
    if result:
        result['acquisition_channel'] = acquisition_channel
        results.append(result)

acquisition_channel_segments = pd.DataFrame(results).set_index('acquisition_channel')
acquisition_channel_segments['significant'] = acquisition_channel_segments['p_value'] < 0.05
print(acquisition_channel_segments.round(2))
print(f"\nSignificant in all acquisition channels: {acquisition_channel_segments['significant'].all()}")
print(f"Positive lift in all acquisition channels: {(acquisition_channel_segments['lift_pp'] > 0).all()}")

                     control_n  treatment_n  control_rate  treatment_rate  \
acquisition_channel                                                         
email                      478          458         14.23           25.33   
organic                   1510         1431         17.35           26.14   
paid_search               1306         1130         16.77           27.08   
referral                  1041          959         18.06           25.23   
social                     753          734         16.07           25.61   

                     lift_pp  p_value  significant  
acquisition_channel                                 
email                  11.10      0.0         True  
organic                 8.78      0.0         True  
paid_search            10.31      0.0         True  
referral                7.18      0.0         True  
social                  9.54      0.0         True  

Significant in all acquisition channels: True
Positive lift in all acquisition channels:

#### 6c. Segment by device

In [8]:
results = []
for device in experiment_results['device'].dropna().unique():
    segment = experiment_results[experiment_results['device'] == device]
    result = run_ztest(segment, 'did_first_transaction')
    if result:
        result['device'] = device
        results.append(result)

device_segments = pd.DataFrame(results).set_index('device')
device_segments['significant'] = device_segments['p_value'] < 0.05
print(device_segments.round(2))
print(f"\nSignificant in all devices: {device_segments['significant'].all()}")
print(f"Positive lift in all devices: {(device_segments['lift_pp'] > 0).all()}")

         control_n  treatment_n  control_rate  treatment_rate  lift_pp  \
device                                                                   
desktop       1649         1558         16.86           25.67     8.82   
mobile        3011         2753         17.00           26.63     9.62   
tablet         431          398         16.94           22.36     5.42   

         p_value  significant  
device                         
desktop     0.00         True  
mobile      0.00         True  
tablet      0.05         True  

Significant in all devices: True
Positive lift in all devices: True


### Test 7: Churn Analysis

In [19]:
# This checks whether churn rate differs by variant (control vs treatment)
churn_result = run_ztest(subscriptions, 'churned')
print("Overall churn by variant:")
print(pd.Series(churn_result).round(4))
#print(f"\nTreatment churn is {abs(churn_result['lift_pp']):.2f}pp lower than control")

if churn_result['p_value'] < 0.05:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

Overall churn by variant:
control_n          876.0000
treatment_n       1248.0000
control_rate        41.0959
treatment_rate      34.3750
lift_pp             -6.7209
p_value              0.0016
dtype: float64
Statistically significant difference


In [16]:
# This checks whether churn rate differs by the plan (free, basic, and premium) within each variant
results = []
for plan in subscriptions['plan'].unique():
    segment = subscriptions[subscriptions['plan'] == plan]
    result = run_ztest(segment, 'churned')
    if result:
        result['plan'] = plan
        results.append(result)

plan_churn = pd.DataFrame(results).set_index('plan')
plan_churn['significant'] = plan_churn['p_value'] < 0.05

print("\nChurn by plan:")
print(plan_churn.round(2))
print(f"\nSignificant in all plans: {plan_churn['significant'].all()}")
print(f"Negative lift (meaning lower churn) in all plans: {(plan_churn['lift_pp'] < 0).all()}")


Churn by plan:
         control_n  treatment_n  control_rate  treatment_rate  lift_pp  \
plan                                                                     
free           436          641         55.05           47.89    -7.15   
basic          302          413         32.45           24.70    -7.75   
premium        138          194         15.94           10.31    -5.63   

         p_value  significant  
plan                           
free        0.02         True  
basic       0.02         True  
premium     0.13        False  

Significant in all plans: False
Negative lift (meaning lower churn) in all plans: True


### Test 8: Revenue Analysis

In [ ]:
# Mann-Whitney test is used here because MRR and LTV are continuous metrics with numeric values, and they do not have yes/no outcomes (like conversion, churn, CTR)
# This test compares whether one group's values tend to be ranked higher than the other group's values overall without assuming any specific distribution shape (like normal/bell curve)


#### 8a. MRR

In [22]:
# For MRR (monthly recurring revenue)
# Even though MRR is numeric, it's not normally distributed as it only takes 3 fixed values (0, 9.99, 24.99) based on the chosen plan, so it's a clustered/discrete-like distribution

control_sub = subscriptions[subscriptions['variant'] == 'control']
treatment_sub = subscriptions[subscriptions['variant'] == 'treatment']

stat, p_value = mannwhitneyu(treatment_sub['mrr'], control_sub['mrr'], alternative='two-sided')

print(f"Control median MRR: {control_sub['mrr'].median():.2f}")
print(f"Treatment median MRR: {treatment_sub['mrr'].median():.2f}")
print(f"Control mean MRR: {control_sub['mrr'].mean():.2f}")
print(f"Treatment mean MRR: {treatment_sub['mrr'].mean():.2f}")
print(f"P-value: {p_value:.4f}")


if p_value < 0.05:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

Control median MRR: 9.99
Treatment median MRR: 0.00
Control mean MRR: 7.38
Treatment mean MRR: 7.19
P-value: 0.5327
No statistically significant difference


#### 8b. LTV

In [23]:
# For LTV (lifetime value)

control_sub = subscriptions[subscriptions['variant'] == 'control']
treatment_sub = subscriptions[subscriptions['variant'] == 'treatment']

stat, p_value = mannwhitneyu(treatment_sub['ltv'], control_sub['ltv'], alternative='two-sided')

print("LTV comparison:")
print(f"Control median LTV: {control_sub['ltv'].median():.2f}")
print(f"Treatment median LTV: {treatment_sub['ltv'].median():.2f}")
print(f"Control mean LTV: {control_sub['ltv'].mean():.2f}")
print(f"Treatment mean LTV: {treatment_sub['ltv'].mean():.2f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Statistically significant difference")
else:
    print("No statistically significant difference")

LTV comparison:
Control median LTV: 0.00
Treatment median LTV: 0.00
Control mean LTV: 153.43
Treatment mean LTV: 162.65
P-value: 0.6127
No statistically significant difference
